# DocGen: Full-Stack AI Document Generator
## Comprehensive Project Documentation & Understanding Report

---

## Part 1: High-Level Architecture Overview
DocGen is an internal tool designed to generate professional, brand-consistent business documents (BRD, FSD, SRS, etc.) using a local LLM (Ollama). 

It operates on a modern, decoupled architecture:
1.  **Frontend:** React 19 + Vite. It uses a clean, single-page application (SPA) model managed by React Router v6. State is managed natively via React Context (no Redux).
2.  **Backend:** Python 3.11+ with FastAPI. It handles routing, authentication, and heavy lifting (talking to the LLM and building files).
3.  **Database:** SQLite via SQLAlchemy. A lightweight, local database to store users, document history, and metadata.
4.  **AI Engine:** Local Ollama running `llama3` or `qwen2.5-coder`. It ensures no corporate data leaves the network.

---

## Part 2: Backend Deep Dive (File by File)

### 2.1 Data Layer (`models/`)
* **`database.py`**: This is the heart of the database. It uses SQLAlchemy to map Python classes to SQLite tables. 
    * *Key Tables:* `User` (auth/roles), `Project` (grouping docs), `Document` (stores history, file paths, generation time), `Comment`, and `Snippet`. 
    * *Advanced Concept:* It includes a migration helper (`_migrate_existing_tables`) using raw `ALTER TABLE` SQL commands to add new columns without losing data if the schema updates.
* **`schemas.py`**: Uses Pydantic. If `database.py` is how data is *stored*, `schemas.py` is how data is *validated* when passing through APIs. It defines request/response shapes (e.g., ensuring a `RegisterRequest` always has an email and password).

### 2.2 Core Business Logic (`services/`)
This is where the actual "work" gets done before being sent to the user.
* **`llm_service.py`**: The bridge to Ollama. It constructs massive prompts by combining the system prompt, brand guide, template, and user instructions.
    * *Advanced Concept:* It implements a `_waf_safe` parser to convert markdown tables into bullet lists to prevent Corporate Web Application Firewalls (WAF) from blocking requests containing pipe `|` characters.
* **`doc_builder.py`**: Converts raw Markdown from the LLM into styled Word documents (`.docx`) using `python-docx` and PDFs using `reportlab`. It manually injects brand colors (e.g., Navy `#1A3C5E`) and replaces placeholders like `[PROJECT NAME]`.
* **`file_parser.py`**: Extracts text from previously uploaded `.docx` or `.pdf` files so the LLM can read them and apply "diff-aware" updates.
* **`auth_service.py`**: Handles JWT (JSON Web Token) generation and validation. It uses `passlib` to securely hash passwords (bcrypt).

### 2.3 API Routing (`routers/`)
These files define the URL endpoints (e.g., `POST /build-doc`).
* **`generate.py`**: The most critical file. It handles the two-step generation process:
    1.  `/preview-doc`: Takes user instructions, hits Ollama, and returns raw Markdown.
    2.  `/build-doc`: Takes the user-approved Markdown, converts it via `doc_builder.py`, saves to SQLite, and streams the `.docx` file back for download.
* **`documents.py`**: Handles fetching document history, updating workflow statuses (Draft -> Approved), and fetching Myers LCS diffs between versions.
* **`auth.py` & `admin.py`**: Manages login, registration, and admin-only settings (like changing templates directly from the web UI).
* **`main.py`**: The entry point. It registers all the routers above and configures CORS (Cross-Origin Resource Sharing) so the React frontend on port `5173` can talk to FastAPI on port `8000`.

---

## Part 3: Frontend Deep Dive (React + Vite)

### 3.1 Routing & State (`src/`)
* **`main.jsx` & `App.jsx`**: Bootstraps the app. `App.jsx` wraps everything in a `<BrowserRouter>` for URL navigation. It uses an `AuthGate` to force unauthenticated users to the Login page.
* **`contexts/AuthContext.jsx`**: The global state manager. It parses the JWT token stored in `localStorage` to determine the user's role (`admin`, `approver`, `author`) and provides an `authHeaders()` function to inject the token into API calls automatically.

### 3.2 The Generation Workflow (`components/`)
* **`DocForm.jsx`**: The starting point. The user selects a document type (e.g., BRD), uploads a previous version (optional), and types instructions. When submitted, it hits the backend `/preview-doc` endpoint.
* **`PreviewPanel.jsx`**: The intermediate step. It receives the Markdown from `DocForm`. The user can read it, edit it inline, and then click "Download", which triggers the `/build-doc` API to get the final `.docx`.
* **`DocHistory.jsx`**: A side panel fetching from `/documents` to show previously generated files.

### 3.3 API Integration (`services/api.js`)
* A clean wrapper around the browser's native `fetch` API. It maps JavaScript functions directly to the FastAPI endpoints, handling JSON parsing and Blob (file) downloads gracefully.

---

## Part 4: Task Progress Report (Manager View)

Below is the chronological breakdown of my technical onboarding and understanding of the DocGen architecture, prepared for today's review.

| Step | Phase Focus | Actions Performed & Concepts Mastered | Status |
| :--- | :--- | :--- | :--- |
| **1** | Architecture & Setup | Analyzed the full-stack blueprint. Understood the interaction between React (Frontend), FastAPI (Backend), SQLite (DB), and local Ollama (LLM). | ✅ Completed |
| **2** | Database Schema | Mapped out SQLAlchemy models (`database.py`) and Pydantic validation schemas. Traced relationships between Users, Projects, and Documents. | ✅ Completed |
| **3** | Auth & Security | Reviewed `auth_service.py` and React `AuthContext.jsx`. Understood JWT lifecycle, bcrypt password hashing, and role-based access control (RBAC). | ✅ Completed |
| **4** | API Layer Mapping | Dissected FastAPI routers. Traced the distinct separation between legacy generation (`/generate-doc`) and the modern two-step workflow (`/preview-doc` -> `/build-doc`). | ✅ Completed |
| **5** | LLM Integration | Analyzed `llm_service.py`. Understood how prompts are dynamically constructed using brand guides and templates, and how the LLM streaming responses are handled. | ✅ Completed |
| **6** | Output Engine | Reviewed `doc_builder.py`. Learned how raw markdown is parsed into XML (`docx`) and formatted into professional PDFs (`reportlab`) with strict brand hex codes. | ✅ Completed |
| **7** | Frontend Workflow | Traced the React component lifecycle. Followed data flow from `DocForm` (input) to `PreviewPanel` (markdown review) to final Blob file download. | ✅ Completed |
| **8** | E2E Synthesis | Connected the dots. Capable of explaining a full request lifecycle from a user clicking "Generate" to the downloaded file and SQLite database record creation. | ✅ Completed |

